# L64 · 打通训练 — 评估与混淆矩阵

**目标**：实现 `train_epoch` 和 `eval_accuracy`，完整训练 KeywordCNN 5 个 epoch，画出损失曲线与混淆矩阵。

🔗 **Aurora 连接**：
- mel 特征由 `aurora.audio.mfcc`（STFT → mel filterbank → log）提供，详见 `src/aurora/audio/`
- 这是 Aurora 项目第一个实际训练完毕的模型，后续推理模块直接加载其权重


## 核心直觉

训练分类器的循环只有三件事：**前向算误差、反向传梯度、优化器更新权重**。交叉熵损失把模型输出的 logit 转成概率再取负对数，越自信且正确则损失越小。每跑完一遍训练集叫一个 epoch；验证集准确率和混淆矩阵告诉我们模型在没见过的数据上表现如何。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## 1. 交叉熵损失 与 Adam 优化器

**交叉熵损失**（`nn.CrossEntropyLoss`）输入原始 logit（形状 `[B, C]`）和整数标签（形状 `[B]`），内部做 softmax 再取负对数似然：

```
loss = -log( exp(logit[y]) / sum_j exp(logit[j]) )
```

不要在模型末尾手动加 softmax——`CrossEntropyLoss` 已经包含了。

**Adam 优化器** 结合了动量（一阶矩）和自适应学习率（二阶矩）。关键词识别任务通常以学习率 `lr=1e-3` 作为起点。

```python
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
```

In [ ]:
# 演示：CrossEntropyLoss 的行为
criterion_demo = nn.CrossEntropyLoss()

# 3 个样本，4 类
logits = torch.tensor([[2.0, 0.5, 0.1, 0.1],   # 自信预测类 0
                        [0.1, 0.1, 2.0, 0.1],   # 自信预测类 2
                        [0.5, 0.5, 0.5, 0.5]])  # 完全不确定
labels = torch.tensor([0, 2, 1])  # 真实标签

loss_val = criterion_demo(logits, labels)
print(f'损失值: {loss_val.item():.4f}')
print('前两个样本预测正确且自信 → 损失小；第三个不确定且错误 → 损失大')

## 2. Per-class Accuracy（每类准确率）

整体准确率 `correct / total` 掩盖了类别不均衡的问题。Per-class accuracy 分别统计每个关键词类别：

```
accuracy_c = (预测为 c 且真实为 c 的样本数) / (真实为 c 的样本总数)
```

如果某类占数据集 1%，整体 99% 准确率可能完全靠忽略它得到；per-class 准确率会暴露这个问题。

In [ ]:
# 演示：手动计算 per-class accuracy
y_true = torch.tensor([0, 0, 1, 1, 2, 2, 2])
y_pred = torch.tensor([0, 1, 1, 1, 0, 2, 2])  # 类0预测错一个，类2预测错一个

num_classes = 3
for c in range(num_classes):
    mask = (y_true == c)
    acc_c = (y_pred[mask] == c).float().mean().item()
    print(f'  类 {c}: {acc_c:.2%}')

overall = (y_pred == y_true).float().mean().item()
print(f'整体准确率: {overall:.2%}')

## 3. 混淆矩阵

混淆矩阵 `C` 的形状为 `[num_classes, num_classes]`：

```
C[i][j] = 真实标签为 i、预测为 j 的样本数
```

- **对角线** `C[i][i]`：正确预测
- **非对角线** `C[i][j]`（i≠j）：错误预测（把类 i 误判为类 j）

用热力图可视化时，颜色越深代表该格样本越多；理想情况下热力图只有对角线是深色的。

In [ ]:
# 演示：用 sklearn 计算并可视化混淆矩阵
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true_np = y_true.numpy()
y_pred_np = y_pred.numpy()
class_names = ['yes', 'no', 'stop']

cm = confusion_matrix(y_true_np, y_pred_np)
print('混淆矩阵:')
print(cm)

fig, ax = plt.subplots(figsize=(4, 3))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('混淆矩阵示例')
plt.tight_layout()
plt.show()

## 4. ✏️ 实现 `train_epoch(model, loader, optimizer, criterion)`

**推理路线**：
1. `model.train()` 开启 dropout/BN 训练模式
2. 遍历 `loader`：`optimizer.zero_grad()` → `out = model(x)` → `loss = criterion(out, y)` → `loss.backward()` → `optimizer.step()`
3. 累积每批 `loss.item()`，最后除以批次数返回 `avg_loss`

**参考输入输出**：
- loader 每批：`x` 形状 `[B, 1, n_mels, T]`，`y` 形状 `[B]`（整数标签）
- 第 1 epoch 典型输出：`avg_loss ≈ 1.5`（4 类随机初始化）；收敛后 `avg_loss < 0.3`

<details><summary>点击查看参考实现</summary>

```python
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)
```

</details>

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    """训练一个 epoch，返回平均损失。"""
    # ✏️ TODO: model.train()
    # ✏️ TODO: for x, y in loader:
    #              optimizer.zero_grad()
    #              out = model(x)
    #              loss = criterion(out, y)
    #              loss.backward()
    #              optimizer.step()
    #              累积 loss.item()
    # ✏️ TODO: return avg_loss
    raise NotImplementedError

In [ ]:
# 目视验证 train_epoch
# 用一个极小的假模型 + 假 loader 快速冒烟测试
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
_x = torch.randn(16, 1, 40, 32)   # 16 个样本，单通道 mel 图
_y = torch.randint(0, 4, (16,))   # 4 类随机标签
_loader = DataLoader(TensorDataset(_x, _y), batch_size=8)

_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(40*32, 4)
)
_opt = torch.optim.Adam(_model.parameters(), lr=1e-3)
_crit = nn.CrossEntropyLoss()

_loss = train_epoch(_model, _loader, _opt, _crit)
assert isinstance(_loss, float), '返回值应为 float'
assert _loss > 0, '损失应大于 0'
print(f'✅ train_epoch 通过，avg_loss = {_loss:.4f}')

## 5. ✏️ 实现 `eval_accuracy(model, loader)`

**推理路线**：
1. `model.eval()` + `torch.no_grad()` 关闭梯度计算，节省显存
2. 遍历 `loader`：`out = model(x)`；`preds = out.argmax(dim=1)`
3. 累积 `(preds == y).sum()` 和总样本数 `total`；返回 `correct / total`

**参考输入输出**：
- 随机初始化 4 类分类器：`accuracy ≈ 0.25`
- 训练 5 epoch 后典型值：`accuracy ≈ 0.75~0.90`（取决于数据量）

<details><summary>点击查看参考实现</summary>

```python
def eval_accuracy(model, loader):
    """在 loader 上评估分类准确率。"""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total
```

</details>

In [ ]:
def eval_accuracy(model, loader):
    """在验证集上计算整体准确率。"""
    # ✏️ TODO: model.eval()
    # ✏️ TODO: with torch.no_grad():
    #              for x, y in loader:
    #                  preds = model(x).argmax(dim=1)
    #                  累积 correct / total
    # ✏️ TODO: return correct / total
    raise NotImplementedError

In [ ]:
# 目视验证 eval_accuracy
_acc_before = eval_accuracy(_model, _loader)
assert 0.0 <= _acc_before <= 1.0, '准确率应在 [0, 1]'
print(f'✅ eval_accuracy 通过，准确率 = {_acc_before:.2%}')

## 6. 参数实验：训练曲线 + 混淆矩阵

**实验参数**：
- `num_epochs = 5`：观察损失从初始值（≈log(num_classes)≈1.39）下降的曲线形状
- `lr = 1e-3`（Adam 默认）：可改为 `1e-4` 对比收敛速度
- `batch_size = 8`：小批量训练，损失曲线会有抖动

**预期现象**：
- 损失曲线前 2 epoch 下降最快，之后趋于平缓
- 混淆矩阵对角线逐渐变深，非对角线变淡
- 若某两类混淆严重（非对角格颜色深），说明它们的 mel 特征相似，可能需要更多训练数据或更深的网络

In [ ]:
# 参数实验：5 epoch 训练曲线 + 混淆矩阵
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 重新初始化模型（用假数据演示，替换为真实 KeywordCNN + DataLoader 即可）
torch.manual_seed(0)
exp_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(40*32, 64),
    nn.ReLU(),
    nn.Linear(64, 4)
)
exp_opt = torch.optim.Adam(exp_model.parameters(), lr=1e-3)
exp_crit = nn.CrossEntropyLoss()

# 假数据 loader（真实使用时替换为 SpeechCommandsDataset 的 DataLoader）
torch.manual_seed(1)
_x_big = torch.randn(128, 1, 40, 32)
_y_big = torch.randint(0, 4, (128,))
train_loader = DataLoader(TensorDataset(_x_big, _y_big), batch_size=16, shuffle=True)
val_loader   = DataLoader(TensorDataset(_x_big, _y_big), batch_size=16)

num_epochs = 5
train_losses = []
val_accs = []

for epoch in range(1, num_epochs + 1):
    loss = train_epoch(exp_model, train_loader, exp_opt, exp_crit)
    acc  = eval_accuracy(exp_model, val_loader)
    train_losses.append(loss)
    val_accs.append(acc)
    print(f'Epoch {epoch}/{num_epochs}  loss={loss:.4f}  acc={acc:.2%}')

# 画训练曲线
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(range(1, num_epochs+1), train_losses, marker='o', color='steelblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('训练损失曲线')
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, num_epochs+1), val_accs, marker='s', color='tomato')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('验证准确率曲线')
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 混淆矩阵
exp_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        all_preds.append(exp_model(xb).argmax(dim=1))
        all_labels.append(yb)
all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

cm = confusion_matrix(all_labels, all_preds)
class_names = ['yes', 'no', 'stop', 'go']

fig2, ax2 = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax2, colorbar=True, cmap='Blues')
ax2.set_title('验证集混淆矩阵（5 epoch 后）')
plt.tight_layout()
plt.show()

## 本课收束

本节实现了 `train_epoch`（返回 `avg_loss: float`）和 `eval_accuracy`（返回 `accuracy: float`），构成了 KeywordCNN 完整的监督训练闭环。mel 特征由 `aurora.audio`（`src/aurora/audio/`）中的手写 STFT → mel filterbank 流水线提供，这是 Aurora 项目第一个端到端训练完毕的神经网络模型。下一节（K4）将在这个已训练模型上做推理：给定一段音频，实时输出关键词概率，并验证延迟指标。